# LayerPassLog Anatomy

Focuses on per-operation records, saved tensors, shapes, parent links, and field display.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerPassLog.source_model_log",
    "tl.LayerPassLog.tensor",
    "tl.LayerPassLog.tensor_dtype",
    "tl.LayerPassLog.tensor_label_raw",
    "tl.LayerPassLog.tensor_memory",
    "tl.LayerPassLog.tensor_memory_str",
    "tl.LayerPassLog.tensor_shape",
    "tl.LayerPassLog.transformed_activation",
    "tl.LayerPassLog.transformed_activation_dtype",
    "tl.LayerPassLog.transformed_activation_memory",
    "tl.LayerPassLog.transformed_activation_shape",
    "tl.LayerPassLog.transformed_gradient",
    "tl.LayerPassLog.transformed_gradient_dtype",
    "tl.LayerPassLog.transformed_gradient_memory",
    "tl.LayerPassLog.transformed_gradient_shape",
    "tl.LayerPassLog.uses_params",
    "tl.ModelLog",
    "tl.ModelLog.DEFAULT_FILL_STATE",
    "tl.ModelLog.FORK_POLICY",
    "tl.ModelLog.PORTABLE_STATE_SPEC",
    "tl.ModelLog._activation_hash_cache",
    "tl.ModelLog._activation_recipe_revision",
    "tl.ModelLog._all_layers_logged",
    "tl.ModelLog._all_layers_saved",
    "tl.ModelLog._append_sequence_id",
    "tl.ModelLog._final_to_raw_layer_labels",
    "tl.ModelLog._gradient_layer_nums_to_save",
    "tl.ModelLog._has_direct_writes",
    "tl.ModelLog._intervention_spec",
    "tl.ModelLog._last_hook_handle_ids",
    "tl.ModelLog._layer_counter",
    "tl.ModelLog._layer_num_to_lookup_keys_dict",
    "tl.ModelLog._layer_nums_to_save",
    "tl.ModelLog._layers_where_internal_branches_merge_with_input",
    "tl.ModelLog._lookup_keys_to_layer_num_dict",
    "tl.ModelLog._pass_finished",
    "tl.ModelLog._raw_layer_dict",
    "tl.ModelLog._raw_layer_labels_list",
    "tl.ModelLog._raw_layer_type_counter",
    "tl.ModelLog._raw_to_final_layer_labels",
    "tl.ModelLog._source_code_blob",
    "tl.ModelLog._source_model_ref",
    "tl.ModelLog._spec_revision",
    "tl.ModelLog._unsaved_layers_lookup_keys",
    "tl.ModelLog._warned_direct_write",
    "tl.ModelLog._warned_mutate_in_place",
    "tl.ModelLog.activation_postfunc",
    "tl.ModelLog.activation_postfunc_repr",
    "tl.ModelLog.activation_transform",
    "tl.ModelLog.add_node_overlay",
    "tl.ModelLog.animate_passes",
    "tl.ModelLog.append_run_state_from",
    "tl.ModelLog.attach_hooks",
    "tl.ModelLog.backward_memory_backend",
    "tl.ModelLog.backward_num_passes",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")